# Codificación de variables categóricas

## Objetivo
Identificar, revisar y transformar las variables categóricas presentes en
`data/processed/dataset_limpio.csv` para preparar las variables predictoras que se utilizarán en el modelo.

De acuerdo a lo anterior se realizaron las siguientes actividades:

- Identificación de variables categóricas y sus valores únicos.
- Revisión de valores faltantes e inconsistencias de formato.
- Definición y aplicación de la codificación adecuada.
- Se mantuvó `Attack_type` separado como variable objetivo (target).
- Se conservó una versión experimental con variables potencialmente leaky codificadas para comparaciones futuras.

## 1. Carga del dataset limpio

El dataset de entrada fue generado en la etapa de limpieza, remitirse a notebooks/03_dataset_limpio.ipynb. Se carga con base a lo especificado en docs/preprocesamiento_limpieza.md en la sección *Instrucciones de uso del dataset limpio*.

In [11]:
import pandas as pd

df = pd.read_csv("../data/processed/dataset_limpio.csv", index_col=0)
print("Dimensiones:", df.shape)
df.head()

Dimensiones: (117922, 37)


,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,flow_pkts_per_sec,down_up_ratio,...,fwd_iat.std,fwd_subflow_bytes,fwd_bulk_bytes,active.std,idle.tot,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,38667,1883,tcp,mqtt,32.011598,9,5,3,0.437341,0.555556,...,1.040307e+07,25.333333,0.0,0.0,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,51143,1883,tcp,mqtt,31.883584,9,5,3,0.439097,0.555556,...,1.046346e+07,25.333333,0.0,0.0,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,44761,1883,tcp,mqtt,32.124053,9,5,3,0.435811,0.555556,...,1.044238e+07,24.666667,0.0,0.0,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,60893,1883,tcp,mqtt,31.961063,9,5,3,0.438033,0.555556,...,1.048253e+07,24.666667,0.0,0.0,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,51087,1883,tcp,mqtt,31.902362,9,5,3,0.438839,0.555556,...,1.044702e+07,25.333333,0.0,0.0,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


## 2. Separación de variables predictoras y variable objetivo

Según las instrucciones de uso del dataset limpio, `Attack_type` es la variable objetivo. 

Para el baseline se excluyen `id.orig_p`, `id.resp_p`, `service` y `proto` dadoq que pueden introducir fuga de información.

In [ ]:
# separar features y variable objetivo
y = df["Attack_type"]
X = df.drop(columns=["Attack_type"])

# variables con posible fuga de información
LEAKY = ["id.orig_p", "id.resp_p", "service", "proto"]

# baseline recomendado
X_sin_leaky = X.drop(columns=LEAKY)

print("Dataset completo:", df.shape)
print("Features originales:", X.shape)
print("Features sin leaky:", X_sin_leaky.shape)
print("Target:", y.shape)

Dataset completo: (117922, 37)
Features originales: (117922, 36)
Features sin leaky: (117922, 32)
Target: (117922,)


## 3. Identificación de variables categóricas

Se identificaron las columnas de tipo texto y se revisaron sus frecuencias,
número de categorías y valores faltantes.

In [12]:
# Identificar columnas categóricas según su tipo de dato
categorical_cols = df.select_dtypes(include=["object", "string", "category", "str"]).columns.tolist()

print("Variables categóricas encontradas:")
print(categorical_cols)

print("\nTipos de datos de las categóricas:")
print(df[categorical_cols].dtypes)

Variables categóricas encontradas:
['proto', 'service', 'Attack_type']

Tipos de datos de las categóricas:
proto          str
service        str
Attack_type    str
dtype: object


In [7]:
for col in categorical_cols:
    print("\n" + "=" * 80)
    print(f"VARIABLE: {col}")
    print(f"Tipo de dato: {df[col].dtype}")
    print(f"Número de valores únicos: {df[col].nunique(dropna=False)}")
    print(f"Valores faltantes: {df[col].isna().sum()}")
    print("\nFrecuencias:")
    print(df[col].value_counts(dropna=False))


VARIABLE: proto
Tipo de dato: str
Número de valores únicos: 3
Valores faltantes: 0

Frecuencias:
proto
tcp     105592
udp      12296
icmp        34
Name: count, dtype: int64

VARIABLE: service
Tipo de dato: str
Número de valores únicos: 10
Valores faltantes: 0

Frecuencias:
service
-         98199
dns        9444
mqtt       4132
http       3289
ssl        2656
ntp         115
dhcp         29
irc          28
ssh          28
radius        2
Name: count, dtype: int64

VARIABLE: Attack_type
Tipo de dato: str
Número de valores únicos: 12
Valores faltantes: 0

Frecuencias:
Attack_type
DOS_SYN_Hping                 90089
Thing_Speak                    7654
ARP_poisioning                 7625
MQTT_Publish                   4142
NMAP_UDP_SCAN                  2584
NMAP_XMAS_TREE_SCAN            2010
NMAP_OS_DETECTION              2000
NMAP_TCP_scan                  1002
DDOS_Slowloris                  533
Wipro_bulb                      219
Metasploit_Brute_Force_SSH       36
NMAP_FIN_SCAN    

## 4. Verificación de inconsistencias en categorías

Se compararon los valores originales con una versión normalizada mediante
eliminación de espacios y conversión a minúsculas. Esta comparación no modifica el dataset y solo se utiliza para detectar categorías duplicadas por formato.

In [8]:
for col in categorical_cols:
    original = df[col].value_counts(dropna=False)
    normalizado = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .value_counts(dropna=False)
    )

    print("\n" + "=" * 80)
    print(f"REVISIÓN DE FORMATO: {col}")
    print("\nValores originales:")
    print(original)
    print("\nValores tras quitar espacios y pasar a minúsculas:")
    print(normalizado)


REVISIÓN DE FORMATO: proto

Valores originales:
proto
tcp     105592
udp      12296
icmp        34
Name: count, dtype: int64

Valores tras quitar espacios y pasar a minúsculas:
proto
tcp     105592
udp      12296
icmp        34
Name: count, dtype: Int64

REVISIÓN DE FORMATO: service

Valores originales:
service
-         98199
dns        9444
mqtt       4132
http       3289
ssl        2656
ntp         115
dhcp         29
irc          28
ssh          28
radius        2
Name: count, dtype: int64

Valores tras quitar espacios y pasar a minúsculas:
service
-         98199
dns        9444
mqtt       4132
http       3289
ssl        2656
ntp         115
dhcp         29
irc          28
ssh          28
radius        2
Name: count, dtype: Int64

REVISIÓN DE FORMATO: Attack_type

Valores originales:
Attack_type
DOS_SYN_Hping                 90089
Thing_Speak                    7654
ARP_poisioning                 7625
MQTT_Publish                   4142
NMAP_UDP_SCAN                  2584
NMAP_XM

### Resultado de la revisión

No se detectaron categorías duplicadas por espacios o diferencias de mayúsculas en `proto` y `service`. En `Attack_type` se observó uso de mayúsculas y minúsculas mixtas, pero cada etiqueta representa una clase distinta; por ello, se conservó el formato original para mantener etiquetas interpretables.

## 5. Estrategia de codificación

| Variable | Rol | Categorías | Método | Justificación |
|---|---|---:|---|---|
| `proto` | Feature experimental | 3 | One-Hot Encoding | Es nominal y no tiene orden natural. |
| `service` | Feature experimental | 10 | One-Hot Encoding | Es nominal y no tiene orden natural. |
| `Attack_type` | Target | 12 | Label Encoding auxiliar | Se conserva como etiqueta original; la versión numérica se genera solo para algoritmos o métricas que la requieran. |

`proto` y `service` están marcadas como potencialmente leaky. Por esta razón,
se codifican para una versión experimental, pero se excluyen del baseline.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# separar target y variables predictoras
y = df["Attack_type"].copy()
X = df.drop(columns=["Attack_type"]).copy()

# columnas categóricas que son features
categorical_features = ["proto", "service"]

# One-hot encoding para variables nominales
X_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

# versión numérica opcional del target
label_encoder = LabelEncoder()
y_encoded = pd.Series(
    label_encoder.fit_transform(y),
    name="Attack_type_encoded",
    index=y.index
)

print("X antes de encoding:", X.shape)
print("X después de encoding:", X_encoded.shape)
print("y original:", y.shape)
print("y codificado:", y_encoded.shape)

print("\nColumnas creadas por One-Hot Encoding:")
print([col for col in X_encoded.columns if col.startswith(("proto_", "service_"))])

print("\nMapa de clases del target:")
for code, label in enumerate(label_encoder.classes_):
    print(f"{code}: {label}")

X antes de encoding: (117922, 36)
X después de encoding: (117922, 47)
y original: (117922,)
y codificado: (117922,)

Columnas creadas por One-Hot Encoding:
['proto_icmp', 'proto_tcp', 'proto_udp', 'service_-', 'service_dhcp', 'service_dns', 'service_http', 'service_irc', 'service_mqtt', 'service_ntp', 'service_radius', 'service_ssh', 'service_ssl']

Mapa de clases del target:
0: ARP_poisioning
1: DDOS_Slowloris
2: DOS_SYN_Hping
3: MQTT_Publish
4: Metasploit_Brute_Force_SSH
5: NMAP_FIN_SCAN
6: NMAP_OS_DETECTION
7: NMAP_TCP_scan
8: NMAP_UDP_SCAN
9: NMAP_XMAS_TREE_SCAN
10: Thing_Speak
11: Wipro_bulb


## 6. Validación del resultado

Se verifica que las variables predictoras codificadas sean completamente numéricas, que no existan valores faltantes y que el número de columnas sea el
esperado. También se revisa el dataset combinado, que conserva la etiqueta original del target.

In [15]:
# validación de las variables predictoras codificadas
print("Dimensiones esperadas de X_encoded: (117922, 47)")
print("Dimensiones obtenidas:", X_encoded.shape)

columnas_no_numericas_X = X_encoded.select_dtypes(
    include=["object", "string", "category", "str"]
).columns.tolist()

print("\n--- Features codificadas: X_encoded ---")
print("Columnas no numéricas:", columnas_no_numericas_X)
print("Valores faltantes:", X_encoded.isna().sum().sum())
print(
    "¿X_encoded quedó completamente numérico?:",
    len(columnas_no_numericas_X) == 0
)

# dataset combinado de features codificadas con target original y target numérico
dataset_encoded = X_encoded.copy()
dataset_encoded["Attack_type"] = y
dataset_encoded["Attack_type_encoded"] = y_encoded

columnas_no_numericas_dataset = dataset_encoded.select_dtypes(
    include=["object", "string", "category", "str"]
).columns.tolist()

print("\n--- Dataset combinado: dataset_encoded ---")
print("Dimensiones:", dataset_encoded.shape)
print("Columnas no numéricas:", columnas_no_numericas_dataset)
print("Valores faltantes:", dataset_encoded.isna().sum().sum())

Dimensiones esperadas de X_encoded: (117922, 47)
Dimensiones obtenidas: (117922, 47)

--- Features codificadas: X_encoded ---
Columnas no numéricas: []
Valores faltantes: 0
¿X_encoded quedó completamente numérico?: True

--- Dataset combinado: dataset_encoded ---
Dimensiones: (117922, 49)
Columnas no numéricas: ['Attack_type']
Valores faltantes: 0


Las variables predictoras codificadas (`X_encoded`) tienen 47 columnas,no presentan valores faltantes y quedaron completamente numéricas. El dataset combinado (`dataset_encoded`) conserva la columna `Attack_type` en formato texto manteniendo una columna no numérica, aunque incluye también `Attack_type_encoded` como representación numérica auxiliar.

## 7. Salidas generadas

Se guardan X_encoded.csv (features codificadas), y_encoded.csv (target numérico auxiliar) y dataset_encoded.csv (features codificadas junto con el target original y su versión numérica). El dataset limpio original no se modifica.

In [8]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

X_encoded.to_csv(output_dir / "X_encoded.csv", index=True)
y_encoded.to_csv(output_dir / "y_encoded.csv", index=True)
dataset_encoded.to_csv(output_dir / "dataset_encoded.csv", index=True)

print("Archivos guardados correctamente en data/processed/")

Archivos guardados correctamente en data/processed/
